<a href="https://colab.research.google.com/github/markmilner21/glazes/blob/main/rough/data/data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This document will be used as EDA/testing image extraction from the data source:

https://www.maycocolors.com/documents/



Upload the files

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

pip install pymupdf pillow

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os

In [ ]:
# Get the uploaded file name
uploaded_file = list(uploaded.keys())[0]
pdf_path = uploaded_file

# Folder to save extracted images
output_folder = "extracted_images"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Open the PDF
doc = fitz.open(pdf_path)

count = 0
for page_number, page in enumerate(doc, start=1):
    for img_index, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        image_ext = base_image["ext"]  # usually png or jpeg

        # Save the image
        image_filename = os.path.join(output_folder, f"page{page_number}_img{img_index}.{image_ext}")
        with open(image_filename, "wb") as f:
            f.write(image_bytes)
        count += 1

print(f"Extracted {count} images to '{output_folder}'")

In [ ]:
import matplotlib.pyplot as plt

sample_images = os.listdir(output_folder)[:10]  # first 5 images
fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))

for ax, img_file in zip(axes, sample_images):
    img = Image.open(os.path.join(output_folder, img_file))
    ax.imshow(img)
    ax.axis("off")
plt.show()

In [ ]:
import os

os.listdir("extracted_images")

images download

In [ ]:
import shutil
shutil.make_archive("extracted_images", 'zip', "extracted_images")
from google.colab import files
files.download("extracted_images.zip")

I also want the type of glaze associated, this will be needed when actually using the tool + allows the code to spot anomolies when processing the data e.g. the instagram logo one

In [ ]:
import fitz
import os

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

for page_number, page in enumerate(doc, start=1):
    text_blocks = page.get_text("blocks")
    image_info = page.get_image_info(hashes=False)
    images = page.get_images(full=True)

    print(f"\n{'='*60}")
    print(f"PAGE {page_number} — {len(images)} images found")
    print(f"{'='*60}")

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        width = base_image["width"]
        height = base_image["height"]

        aspect_ratio = width / height if height > 0 else 0
        is_square = 0.8 <= aspect_ratio <= 1.2

        bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)

        if bbox is None:
            label = "unsure (no bbox)"
        else:
            ix0, iy0, ix1, iy1 = bbox
            img_cx = (ix0 + ix1) / 2

            closest_text = "unsure"
            min_distance = float("inf")

            for block in text_blocks:
                bx0, by0, bx1, by1, btext, *_ = block
                btext = btext.strip()
                if not btext:
                    continue
                if by0 >= iy1:
                    b_cx = (bx0 + bx1) / 2
                    if ix0 - 20 <= b_cx <= ix1 + 20:
                        distance = by0 - iy1
                        if distance < min_distance:
                            min_distance = distance
                            closest_text = btext

            if closest_text == "unsure":
                for block in text_blocks:
                    bx0, by0, bx1, by1, btext, *_ = block
                    btext = btext.strip()
                    if not btext or by0 < iy1:
                        continue
                    distance = by0 - iy1
                    if distance < min_distance:
                        min_distance = distance
                        closest_text = btext

            label = closest_text.replace("\n", " ")[:60]

        status = "✅ ACCEPTED" if is_square else "❌ REJECTED"
        print(f"  [{img_index}] {status} | {width}x{height} (ratio {aspect_ratio:.2f}) | label: {label}")

In [ ]:
import fitz
import io
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

for page_number, page in enumerate(doc, start=1):
    text_blocks = page.get_text("blocks")
    image_info = page.get_image_info(hashes=False)
    images = page.get_images(full=True)

    if not images:
        continue

    print(f"\n{'='*60}")
    print(f"PAGE {page_number} — {len(images)} images")
    print(f"{'='*60}")

    cols = 6
    rows = -(-len(images) // cols)  # ceiling division
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        width = base_image["width"]
        height = base_image["height"]

        aspect_ratio = width / height if height > 0 else 0
        is_square = 0.8 <= aspect_ratio <= 1.2

        bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)
        closest_text = "unsure"

        if bbox:
            ix0, iy0, ix1, iy1 = bbox
            min_distance = float("inf")

            for block in text_blocks:
                bx0, by0, bx1, by1, btext, *_ = block
                btext = btext.strip()
                if not btext:
                    continue
                if by0 >= iy1:
                    b_cx = (bx0 + bx1) / 2
                    if ix0 - 20 <= b_cx <= ix1 + 20:
                        distance = by0 - iy1
                        if distance < min_distance:
                            min_distance = distance
                            closest_text = btext

            if closest_text == "unsure":
                min_distance = float("inf")
                for block in text_blocks:
                    bx0, by0, bx1, by1, btext, *_ = block
                    btext = btext.strip()
                    if not btext or by0 < iy1:
                        continue
                    distance = by0 - iy1
                    if distance < min_distance:
                        min_distance = distance
                        closest_text = btext

        pil_img = Image.open(io.BytesIO(image_bytes))
        ax = axes[img_index]
        ax.imshow(pil_img)
        ax.axis("off")

        color = "green" if is_square else "red"
        status = "✅" if is_square else "❌"
        label_display = closest_text.replace("\n", " ")[:30]
        ax.set_title(f"{status} {label_display}", fontsize=7, color=color, wrap=True)

    # Hide unused axes
    for j in range(len(images), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("accepted", 'zip', "accepted")
shutil.make_archive("rejected", 'zip', "rejected")

files.download("accepted.zip")
files.download("rejected.zip")

the naming of these files isn't working just yet, let's try and work with that

this will probably also likely allow us to get rid of more images slipping through the crack as we currently only filter based on aspect ratio

In [ ]:
import fitz
import io
import re
from PIL import Image
import matplotlib.pyplot as plt

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

def find_label(bbox, text_blocks):
    ix0, iy0, ix1, iy1 = bbox
    closest_text = None
    min_distance = float("inf")

    for block in text_blocks:
        bx0, by0, bx1, by1, btext, *_ = block
        btext = btext.strip()
        if not btext:
            continue
        if by0 >= iy1:
            b_cx = (bx0 + bx1) / 2
            if ix0 - 20 <= b_cx <= ix1 + 20:
                distance = by0 - iy1
                if distance < min_distance:
                    min_distance = distance
                    closest_text = btext

    # Fallback: nearest below regardless of horizontal position
    if not closest_text:
        min_distance = float("inf")
        for block in text_blocks:
            bx0, by0, bx1, by1, btext, *_ = block
            btext = btext.strip()
            if not btext or by0 < iy1:
                continue
            distance = by0 - iy1
            if distance < min_distance:
                min_distance = distance
                closest_text = btext

    return closest_text or "unsure"

def is_glaze_label(text):
    """Check if text contains a glaze code like EL-107, EL-212 etc."""
    return bool(re.search(r'EL[-–]\d+', text, re.IGNORECASE))

for page_number, page in enumerate(doc, start=1):
    text_blocks = page.get_text("blocks")
    image_info = page.get_image_info(hashes=False)
    images = page.get_images(full=True)

    if not images:
        continue

    print(f"\n{'='*60}")
    print(f"PAGE {page_number} — {len(images)} images")
    print(f"{'='*60}")

    cols = 6
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten()

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        width = base_image["width"]
        height = base_image["height"]

        if height == 0:
            continue

        aspect_ratio = width / height
        is_square = 0.8 <= aspect_ratio <= 1.2

        bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)
        label = find_label(bbox, text_blocks) if bbox else "unsure"
        has_glaze_code = is_glaze_label(label)

        # Accept only if square AND has a valid EL- glaze code nearby
        accepted = is_square and has_glaze_code

        pil_img = Image.open(io.BytesIO(image_bytes))
        ax = axes[img_index]
        ax.imshow(pil_img)
        ax.axis("off")

        if accepted:
            status = "✅"
            color = "green"
        elif is_square and not has_glaze_code:
            status = "⚠️"   # square but no glaze code
            color = "orange"
        else:
            status = "❌"
            color = "red"

        label_display = label.replace("\n", " ")[:30]
        ax.set_title(f"{status} {label_display}", fontsize=7, color=color, wrap=True)

    for j in range(len(images), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import fitz

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

page = doc[1]  # page 2 has the swatches
text_blocks = page.get_text("blocks")
image_info = page.get_image_info(hashes=False)
images = page.get_images(full=True)

# Just check first 5 images
for img_index in range(min(5, len(images))):
    bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)
    if not bbox:
        print(f"[{img_index}] no bbox")
        continue

    ix0, iy0, ix1, iy1 = bbox
    print(f"\n[{img_index}] image bbox: {bbox}")
    print(f"  Text blocks below:")

    for block in text_blocks:
        bx0, by0, bx1, by1, btext, *_ = block
        if by0 >= iy1:
            b_cx = (bx0 + bx1) / 2
            if ix0 - 20 <= b_cx <= ix1 + 20:
                print(f"    repr: {repr(btext)}")

In [ ]:
import fitz
import io
from PIL import Image
import matplotlib.pyplot as plt

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

page = doc[1]  # page 2
image_info = page.get_image_info(hashes=False)
images = page.get_images(full=True)

fig, axes = plt.subplots(1, 5, figsize=(15, 4))

for img_index in range(min(5, len(images))):
    xref = images[img_index][0]
    base_image = doc.extract_image(xref)
    pil_img = Image.open(io.BytesIO(base_image["image"]))

    bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)

    axes[img_index].imshow(pil_img)
    axes[img_index].axis("off")
    axes[img_index].set_title(f"[{img_index}]\nbbox: {bbox[1]:.0f},{bbox[3]:.0f}" if bbox else f"[{img_index}] no bbox", fontsize=8)

plt.tight_layout()
plt.show()

My guess is the text in this PDF is actually embedded as part of the images rather than as real selectable text — which would explain why no blocks are found. The output will confirm this.

In [ ]:
import fitz

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

page = doc[1]

# Try all different text extraction methods
print("=== get_text('blocks') ===")
blocks = page.get_text("blocks")
print(f"Block count: {len(blocks)}")
for b in blocks[:3]:
    print(repr(b))

print("\n=== get_text('words') ===")
words = page.get_text("words")
print(f"Word count: {len(words)}")
for w in words[:5]:
    print(repr(w))

print("\n=== get_text('raw') first 500 chars ===")
raw = page.get_text("raw")
print(repr(raw[:500]))

The text is baked into the images — there's no selectable text in this PDF at all. We'll need OCR to read the labels. Let's use pytesseract:

In [ ]:
import subprocess
subprocess.run(["apt-get", "install", "-y", "tesseract-ocr"], capture_output=True)
subprocess.run(["pip", "install", "pytesseract"], capture_output=True)

this is proving slightly more difficult so let's come back to this

we are going to try:
1. Position clustering — swatches appear in a grid pattern, so images whose positions don't cluster with others on the page are likely not swatches
2. Colour variance filter — glaze swatches have rich colour variation across the image. Logos and icons tend to be mostly white/transparent. We could reject images whose pixel variance is below a threshold.

In [ ]:
import fitz
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

def colour_variance(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    arr = np.array(img, dtype=float)
    return float(np.std(arr))

def is_in_cluster(bbox, all_bboxes, min_neighbours=2, max_dist=200, size_tolerance=5):
    ix0, iy0, ix1, iy1 = bbox
    iw, ih = ix1 - ix0, iy1 - iy0
    neighbours = 0
    for other in all_bboxes:
        if other == bbox:
            continue
        ox0, oy0, ox1, oy1 = other
        ow, oh = ox1 - ox0, oy1 - oy0
        size_match = abs(iw - ow) < size_tolerance and abs(ih - oh) < size_tolerance
        dist = ((((ix0+ix1)/2) - ((ox0+ox1)/2))**2 + (((iy0+iy1)/2) - ((oy0+oy1)/2))**2)**0.5
        if size_match and dist < max_dist:
            neighbours += 1
    return neighbours >= min_neighbours

VARIANCE_THRESHOLD = 21
ASPECT_MIN, ASPECT_MAX = 0.8, 1.2

for page_number, page in enumerate(doc, start=1):
    image_info = page.get_image_info(hashes=False)
    images = page.get_images(full=True)

    if not images:
        continue

    all_bboxes = [i["bbox"] for i in image_info if i.get("bbox")]

    print(f"\n{'='*60}")
    print(f"PAGE {page_number} — {len(images)} images")
    print(f"{'='*60}")

    cols = 6
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes = axes.flatten()

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        width = base_image["width"]
        height = base_image["height"]

        if height == 0:
            axes[img_index].axis("off")
            continue

        aspect_ratio = width / height
        is_square = ASPECT_MIN <= aspect_ratio <= ASPECT_MAX

        bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)
        in_cluster = is_in_cluster(bbox, all_bboxes) if bbox else False

        variance = colour_variance(image_bytes)
        has_variance = variance >= VARIANCE_THRESHOLD

        accepted = is_square and in_cluster and has_variance

        pil_img = Image.open(io.BytesIO(image_bytes))
        ax = axes[img_index]
        ax.imshow(pil_img)
        ax.axis("off")

        # Build detailed per-condition report
        shape_line  = f"{'✅' if is_square    else '❌'} ratio={aspect_ratio:.2f} (need {ASPECT_MIN}-{ASPECT_MAX})"
        cluster_line = f"{'✅' if in_cluster  else '❌'} cluster={'yes' if in_cluster else 'no'}"
        var_line    = f"{'✅' if has_variance else '❌'} var={variance:.0f} (need >={VARIANCE_THRESHOLD})"

        overall = "✅ ACCEPTED" if accepted else "❌ REJECTED"
        color = "green" if accepted else "red"

        title = f"{overall}\n{shape_line}\n{cluster_line}\n{var_line}"
        ax.set_title(title, fontsize=6.5, color=color, loc="left")

    for j in range(len(images), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import fitz
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

def colour_variance(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    arr = np.array(img, dtype=float)
    return float(np.std(arr))

def is_in_cluster(bbox, all_bboxes, min_neighbours=2, max_dist=200, size_tolerance=5):
    ix0, iy0, ix1, iy1 = bbox
    iw, ih = ix1 - ix0, iy1 - iy0
    neighbours = 0
    for other in all_bboxes:
        if other == bbox:
            continue
        ox0, oy0, ox1, oy1 = other
        ow, oh = ox1 - ox0, oy1 - oy0
        size_match = abs(iw - ow) < size_tolerance and abs(ih - oh) < size_tolerance
        dist = ((((ix0+ix1)/2) - ((ox0+ox1)/2))**2 + (((iy0+iy1)/2) - ((oy0+oy1)/2))**2)**0.5
        if size_match and dist < max_dist:
            neighbours += 1
    return neighbours >= min_neighbours

VARIANCE_THRESHOLD = 21
ASPECT_MIN, ASPECT_MAX = 0.8, 1.2

for page_number, page in enumerate(doc, start=1):
    image_info = page.get_image_info(hashes=False)
    images = page.get_images(full=True)

    if not images:
        continue

    all_bboxes = [i["bbox"] for i in image_info if i.get("bbox")]

    print(f"\n{'='*60}")
    print(f"PAGE {page_number} — {len(images)} images")
    print(f"{'='*60}")

    cols = 6
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes = axes.flatten()

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        width = base_image["width"]
        height = base_image["height"]

        if height == 0:
            axes[img_index].axis("off")
            continue

        aspect_ratio = width / height
        is_square = ASPECT_MIN <= aspect_ratio <= ASPECT_MAX

        bbox = next((i["bbox"] for i in image_info if i["number"] == img_index), None)
        in_cluster = is_in_cluster(bbox, all_bboxes) if bbox else False

        variance = colour_variance(image_bytes)
        has_variance = variance >= VARIANCE_THRESHOLD

        accepted = is_square and in_cluster and has_variance

        pil_img = Image.open(io.BytesIO(image_bytes))
        ax = axes[img_index]
        ax.imshow(pil_img)
        ax.axis("off")

        # Build detailed per-condition report
        shape_line  = f"{'✅' if is_square    else '❌'} ratio={aspect_ratio:.2f} (need {ASPECT_MIN}-{ASPECT_MAX})"
        cluster_line = f"{'✅' if in_cluster  else '❌'} cluster={'yes' if in_cluster else 'no'}"
        var_line    = f"{'✅' if has_variance else '❌'} var={variance:.0f} (need >={VARIANCE_THRESHOLD})"

        overall = "✅ ACCEPTED" if accepted else "❌ REJECTED"
        color = "green" if accepted else "red"

        title = f"{overall}\n{shape_line}\n{cluster_line}\n{var_line}"
        ax.set_title(title, fontsize=6.5, color=color, loc="left")

    for j in range(len(images), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

The problem is clear — get_images() is returning the entire grid as one big image rather than individual swatches. This is a fundamental PDF structure issue; the swatches are embedded as a single large image, not individual ones.
The right approach here is to render the whole page and then slice it into a grid rather than trying to extract individual embedded images. But to do that reliably we'd need to either:

Manual grid detection — render the page, detect the grid lines using OpenCV, then slice each cell
Fixed grid slicing — if the grid is always the same layout (e.g. 4x6), just divide the rendered page into equal tiles
Give up on PDF image extraction entirely — render each page as a high-res image and use OpenCV to detect individual square regions by their borders/edges

Given the inconsistency across different PDFs, option 3 with OpenCV edge detection is probably the most robust. It would find any rectangular region that looks like a swatch regardless of how the PDF structured it internally.

In [ ]:
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import io

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

def colour_variance(img_array):
    return float(np.std(img_array.astype(float)))

VARIANCE_THRESHOLD = 21
ASPECT_MIN, ASPECT_MAX = 0.8, 1.2
MIN_SIZE = 50   # minimum pixel width/height to ignore tiny icons
MAX_SIZE = 400  # maximum pixel width/height to ignore full page images

for page_number, page in enumerate(doc, start=1):
    # Render page to high-res image
    pix = page.get_pixmap(dpi=150)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    # Convert to grayscale and find edges
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)

    # Dilate edges to close small gaps
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)

    # Find contours
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < MIN_SIZE or h < MIN_SIZE:
            continue
        if w > MAX_SIZE or h > MAX_SIZE:
            continue
        aspect = w / h
        if not (ASPECT_MIN <= aspect <= ASPECT_MAX):
            continue
        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)
        if variance <= VARIANCE_THRESHOLD:
            continue
        candidates.append((x, y, w, h, variance))

    # Remove duplicates — keep largest when boxes heavily overlap
    def overlap(a, b, threshold=0.5):
        ax, ay, aw, ah, _ = a
        bx, by, bw, bh, _ = b
        ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
        iy = max(0, min(ay+ah, by+bh) - max(ay, by))
        inter = ix * iy
        smaller = min(aw * ah, bw * bh)
        return inter / smaller > threshold if smaller > 0 else False

    filtered = []
    for c in sorted(candidates, key=lambda x: x[2]*x[3], reverse=True):
        if not any(overlap(c, kept) for kept in filtered):
            filtered.append(c)

    print(f"\nPage {page_number} — {len(filtered)} swatches detected")

    if not filtered:
        continue

    cols = 6
    rows = -(-len(filtered) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten()

    for i, (x, y, w, h, variance) in enumerate(sorted(filtered, key=lambda c: (c[1], c[0]))):
        crop = img_array[y:y+h, x:x+w]
        axes[i].imshow(crop)
        axes[i].axis("off")
        axes[i].set_title(f"var={variance:.0f}\n{w}x{h}", fontsize=7)

    for j in range(len(filtered), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"Page {page_number}", fontsize=10)
    plt.tight_layout()
    plt.show()

170 x 170 seems accurate

In [ ]:
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

TARGET_SIZE = 170
SIZE_TOLERANCE = 5
VARIANCE_THRESHOLD = 21

def colour_variance(crop):
    return float(np.std(crop.astype(float)))

def overlap(a, b, threshold=0.5):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix * iy
    smaller = min(aw * ah, bw * bh)
    return inter / smaller > threshold if smaller > 0 else False

def ocr_strip(img_array, x, y, w, h):
    strip_y1 = y + h
    strip_y2 = min(y + h + 60, img_array.shape[0])
    strip = img_array[strip_y1:strip_y2, max(0, x-10):min(img_array.shape[1], x+w+10)]
    strip_pil = Image.fromarray(strip)
    strip_pil = strip_pil.resize((strip_pil.width * 3, strip_pil.height * 3), Image.LANCZOS)
    label = pytesseract.image_to_string(strip_pil, config='--psm 6').strip()
    return label.replace("\n", " ")[:50]

def detect_swatches(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        # Strict size filter — must be within +-5px of 170x170
        if abs(w - TARGET_SIZE) > SIZE_TOLERANCE:
            continue
        if abs(h - TARGET_SIZE) > SIZE_TOLERANCE:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)
        if variance <= VARIANCE_THRESHOLD:
            continue

        candidates.append((x, y, w, h, variance))

    # Deduplicate overlapping boxes
    filtered = []
    for c in sorted(candidates, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    # Sort top-to-bottom, left-to-right
    return sorted(filtered, key=lambda c: (c[1], c[0]))

accepted_swatches = []  # (page, crop, label)
rejected_swatches = []

for page_number, page in enumerate(doc, start=1):
    pix = page.get_pixmap(dpi=150)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    swatches = detect_swatches(img_array)
    print(f"Page {page_number} — {len(swatches)} swatches detected")

    if not swatches:
        continue

    cols = 6
    rows = -(-len(swatches) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten()

    for i, (x, y, w, h, variance) in enumerate(swatches):
        crop = img_array[y:y+h, x:x+w]
        label = ocr_strip(img_array, x, y, w, h)

        axes[i].imshow(crop)
        axes[i].axis("off")
        axes[i].set_title(f"var={variance:.0f}\n{label}", fontsize=6, color="green")

        accepted_swatches.append((page_number, crop, label))

    for j in range(len(swatches), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"Page {page_number} — {len(swatches)} swatches", fontsize=10)
    plt.tight_layout()
    plt.show()

print(f"\nTotal accepted: {len(accepted_swatches)}")

that seems to cut a bit too deep
The only change to the filter is abs(w - h) > 3 — so instead of comparing against a fixed 170px target, it just requires width and height to be within 3px of each other. This means it'll work on any PDF where swatches are square regardless of their actual size.

In [ ]:
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

VARIANCE_THRESHOLD = 21
MIN_SIZE = 150
MAX_SIZE = 185

def colour_variance(crop):
    return float(np.std(crop.astype(float)))

def overlap(a, b, threshold=0.5):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix * iy
    smaller = min(aw * ah, bw * bh)
    return inter / smaller > threshold if smaller > 0 else False

def detect_swatches(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted = []
    rejected = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        reasons = []
        if not (MIN_SIZE <= w <= MAX_SIZE and MIN_SIZE <= h <= MAX_SIZE):
            reasons.append(f"size {w}x{h}")
        if abs(w - h) > 3:
            reasons.append(f"not square (diff={abs(w-h)})")
        if variance <= VARIANCE_THRESHOLD:
            reasons.append(f"flat var={variance:.0f}")

        if reasons:
            rejected.append((x, y, w, h, variance, ", ".join(reasons), crop))
        else:
            accepted.append((x, y, w, h, variance))

    filtered = []
    for c in sorted(accepted, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    return sorted(filtered, key=lambda c: (c[1], c[0])), rejected

for page_number, page in enumerate(doc, start=1):
    pix = page.get_pixmap(dpi=150)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    accepted, rejected = detect_swatches(img_array)
    print(f"Page {page_number} — {len(accepted)} accepted, {len(rejected)} rejected")

    if accepted:
        cols = 6
        rows = -(-len(accepted) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = axes.flatten()
        for i, (x, y, w, h, variance) in enumerate(accepted):
            axes[i].imshow(img_array[y:y+h, x:x+w])
            axes[i].axis("off")
            axes[i].set_title(f"var={variance:.0f}", fontsize=7, color="green")
        for j in range(len(accepted), len(axes)):
            axes[j].axis("off")
        plt.suptitle(f"Page {page_number} — ✅ ACCEPTED ({len(accepted)})", fontsize=10, color="green")
        plt.tight_layout()
        plt.show()

    if rejected:
        cols = 6
        rows = -(-len(rejected) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = axes.flatten()
        for i, (x, y, w, h, variance, reason, crop) in enumerate(rejected):
            axes[i].imshow(crop)
            axes[i].axis("off")
            axes[i].set_title(f"{reason}", fontsize=6, color="red")
        for j in range(len(rejected), len(axes)):
            axes[j].axis("off")
        plt.suptitle(f"Page {page_number} — ❌ REJECTED ({len(rejected)})", fontsize=10, color="red")
        plt.tight_layout()
        plt.show()